In [15]:
from utils_basic import (
    copy_hamiltonian
)
from utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state,
    create_composite_state
)
from utils_m1_seniority import (
    project_out_seniority_symmetries
)
from utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn_vo,
    factorize_state,
    evaluate_fully_classical_factors
)
from utils_m3_swap import (
    XorY_augment
)
from utils_m4_partitioning import (
    sorted_insertion_decomposition
)
from utils_results import (
    variance_of_decomp,
    sampling_cost
)
from openfermion import (
    get_sparse_operator,
    jordan_wigner
)

import numpy as np
import pickle
import sys


def qubit_hamiltonian_1norm(Hqub):
    Hcopy = copy_hamiltonian(Hqub)
    # Hcopy -= Hcopy.constant
    Hcopy.compress()

    l1norm = 0
    for term, coef in Hcopy.terms.items():
        l1norm += np.abs(coef)
    return l1norm

def inclusive_upper_triangle_array(X):
    L = []
    N = X.shape[0]
    for i in range(N):
        for j in range(N):
            if i >= j:
                L.append(X[i,j])
    return np.array(L)

In [16]:
# load Q-SENSE basis states

molecule        = 'n2'
bond_length     = 1.0
filename        = f'{molecule}_data/UCSF_sym_comp_for_Praveen_Smik_{bond_length}.dump'
output_filename = f'main_outputs/{molecule}_{bond_length}_VO_serial'

with open(filename, 'rb') as f:
    (
    CSF_tz_states,
    W_amplitudes,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    UCSF_tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

# rotate orbitals and obtain Hamiltonian operator

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)
Hqub   -= Hqub.constant
Hqub.compress()

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

# process information so that we can taper and factorize the Q-SENSE states

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn_vo(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list  = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize the Q-SENSE basis states

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) 
                                   for i in range(Nstates)]

Nterms_full = len(Hqub.terms)
L1norm_full = qubit_hamiltonian_1norm(Hqub)

In [17]:
L1norm_mat = np.zeros([Nstates, Nstates])
Nterms_mat = np.zeros([Nstates, Nstates])

for i in range(Nstates):
    print(i, i, end='\r')
    ket_f      = factorized_tapered_statevectors[i]
    ket_labels = UCSF_information[i][0]
    ket_config = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    HQ, ketQ, _, NQ = evaluate_fully_classical_factors(ket_f, ket_f, ket_labels, ket_labels, Htapered)

    HQ -= HQ.constant
    HQ.compress()
    Nterms_mat[i,i] = len(HQ.terms)
    L1norm_mat[i,i] = qubit_hamiltonian_1norm(HQ)

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(i, j, end='\r')

            bra_f              = factorized_tapered_statevectors[i]
            bra_labels         = UCSF_information[i][0]
            bra_config         = configs[i]

            ket_f              = factorized_tapered_statevectors[j]
            ket_labels         = UCSF_information[j][0]
            ket_config         = configs[j]

            Htapered           = project_out_seniority_symmetries(Hqub, Nqubits, bra_config, ket_config)
            HQ, braQ, ketQ, NQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)

            if NQ != 0:
                Nterms_mat[i,j] = len(HQ.terms)
                Nterms_mat[j,i] = len(HQ.terms)
                L1norm_mat[i,j] = qubit_hamiltonian_1norm(HQ)
                L1norm_mat[j,i] = qubit_hamiltonian_1norm(HQ)

In [18]:
Nterms_list = inclusive_upper_triangle_array(Nterms_mat)
L1norm_list = inclusive_upper_triangle_array(L1norm_mat)

Rterms_list  = Nterms_list / Nterms_full
Rlambda_list = L1norm_list / L1norm_full

print(f'''
    Max Terms : {np.max(Rterms_list)}
    Avg Terms : {np.average(Rterms_list)}
    Max Lamda : {np.max(Rlambda_list)}
    Avg Lamda : {np.average(Rlambda_list)}
''')


    Max Terms : 0.05547054322876817
    Avg Terms : 0.014729771686459753
    Max Lamda : 0.8053547433398214
    Avg Lamda : 0.05143454738088138



This version is different from `main_1norm_terms.ipynb` only in that the constant term of $H$ is not used when calculating the number of terms (denominator of the Nterms ratio).

H2O

```
Max Terms : 0.06451612903225806
Avg Terms : 0.03153190895126379
Max Lamda : 0.7611545514311772
Avg Lamda : 0.059411374260790954
```

N2

```
Max Terms : 0.05547054322876817
Avg Terms : 0.014729771686459753
Max Lamda : 0.8053547433398214
Avg Lamda : 0.05143454738088138
```